## Experiment 1: Robust VRP Extraction

> **Paper:** *The VIX, the Variance Premium and Stock Market Volatility*  
> Geert Bekaert & Marie Hoerova, ECB Working Paper No. 1675, May 2014

---

## Objective

Isolate the continuous Variance Risk Premium by netting option-implied variance against a physical conditional variance forecast derived from a daily Corsi HAR model, and validate the predictive engine's stability using an end-of-day loop with rolling-window re-estimation.

The pipeline evaluation compares three benchmarks:

| Benchmark | Description |
|-----------|-------------|
| **Paper (B&H 2014)** | VIX²/12 + 5-min intraday RV, 1990–2010 |
| **Static HAR baseline** | VIX²/12 + daily-sq RV, one-time OLS fit, 75% OOS split |
| **Production loop (rolling)** | VIX²/12 + daily-sq RV, 500-day rolling OLS, zero look-ahead |

### Steps

| Step | Description |
|------|-------------|
| **1** | Implied Variance Computation — $IV_t = VIX_t^2 / 12$ |
| **2** | Physical Realized Variance Processing |
| **3** | Corsi HAR Panel Fitting (Bekaert & Hoerova, 2014) |
| **4** | Variance Risk Premium Isolation — $VP_t = IV_t - CV_t$ |
| **5** | End-of-Day Production Loop — 500-day rolling OLS, zero look-ahead |
| **6** | Statistical Accuracy Evaluation (Bekaert & Hoerova, 2014) |

---
## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# ── Path setup ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(".").resolve()
if (NOTEBOOK_DIR / "experiment1 - VRP Computation").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "experiment1 - VRP Computation"

ROOT   = NOTEBOOK_DIR.parent
BH_DIR = ROOT / "bh_replication"
DATA   = ROOT / "data"
OUTPUT = NOTEBOOK_DIR / "output"

sys.path.insert(0, str(BH_DIR))

from data_prep import load_sp500_returns, compute_rv_components
from har_model  import estimate_har, out_of_sample_forecast, NW_LAGS, _nw_se

%matplotlib inline
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True,
                      "grid.alpha": 0.3, "font.size": 10})

print(f"Notebook dir : {NOTEBOOK_DIR}")
print(f"Output dir   : {OUTPUT}")

In [ ]:
def compute_metrics(y_actual, y_hat, label=""):
    valid = ~(np.isnan(y_actual) | np.isnan(y_hat))
    ya, yh = y_actual[valid], y_hat[valid]
    err  = ya - yh
    rmse = float(np.sqrt((err ** 2).mean()))
    mae  = float(np.abs(err).mean())
    mape = float((np.abs(err) / np.clip(ya, 0.01, None)).mean())
    mz   = OLS(ya, add_constant(yh)).fit()
    return {"label": label, "n": int(valid.sum()),
            "mz_r2": round(float(mz.rsquared), 4),
            "rmse":  round(rmse, 3),
            "mae":   round(mae,  3),
            "mape":  round(mape, 4)}

In [ ]:
# ── Paper benchmarks (Table 3, Model 8) ──────────────────────────────────────
PAPER_COEFS = {
    "const":    3.730,
    "VIX2_lag": 0.108,   # α  (IV_t coefficient)
    "RV22_lag": 0.199,   # β^m
    "RV5_lag":  0.330,   # β^w
    "RV1_lag":  0.107,   # β^d
}
PAPER_OOS    = {"rmse": 46.077, "mae": 16.856, "mape": 0.347, "mz_r2": 0.555}
PAPER_IS_RMSE = 10.508

ROLL_WIN = 500   # production-loop rolling window (trading days)

print("Paper benchmarks (B&H 2014, Table 3, Model 8):")
print(f"  IS RMSE = {PAPER_IS_RMSE}")
print(f"  OOS MZ-R² = {PAPER_OOS['mz_r2']}  RMSE = {PAPER_OOS['rmse']}  "
      f"MAE = {PAPER_OOS['mae']}  MAPE = {PAPER_OOS['mape']}")

---
## Step 1 — Implied Variance Computation

Two pure implied variance series are computed and compared throughout:

| Series | Formula | Source | Available from |
|--------|---------|--------|---------------|
| **VIX-based** | $IV_t = VIX_t^2 / 12$ | CBOE VIX index | Jan 1990 |
| **VS-based**  | $IV_t = VS_t^2 / 12$  | SPX 1-month variance swap | Nov 2008 |

Neither series uses the other as a fallback. The VIX-based series drives the main production loop (full history). The VS-based series runs a parallel loop from 2008 onwards for direct comparison.

In [ ]:
from data_prep import load_vix

# ── VIX-based implied variance (full history) ─────────────────────────────────
vix  = load_vix()
ivar = vix ** 2 / 12.0
ivar.name = "IVar"

# ── VS-based implied variance (pure, no fallback, 2008 onwards) ───────────────
vs_raw = pd.read_csv(DATA / "EquityIndexVarianceSwapData.csv", parse_dates=["DATE"])
spx1m  = (vs_raw[(vs_raw["UNDERLYING"] == "SPX") & (vs_raw["TENOR_MONTHS"] == 1.0)]
          .sort_values("DATE").set_index("DATE")["IMPLIED_VOLATILITY"])
spx1m.index.name = "date"
ivar_vs = spx1m ** 2 / 12.0
ivar_vs.name = "IVar_VS"

print(f"VIX-based IV_t : {len(ivar):,} obs  ({ivar.index.min().date()} - {ivar.index.max().date()})")
print(f"VS-based  IV_t : {len(ivar_vs):,} obs  ({ivar_vs.index.min().date()} - {ivar_vs.index.max().date()})")

# Overlap stats
ol_start = ivar_vs.dropna().index.min()
vix_ol = ivar.loc[ol_start:]
vs_ol  = ivar_vs.loc[ol_start:]
print(f"\nOverlap period ({ol_start.date()} onwards):")
print(f"  VIX IVar mean={vix_ol.mean():.3f}  VS IVar mean={vs_ol.mean():.3f}  Ratio={vix_ol.mean()/vs_ol.mean():.4f}")
print(f"  Correlation: {vix_ol.corr(vs_ol.reindex(vix_ol.index)):.4f}")

# ── Plot both series ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ax = axes[0]
ax.plot(ivar.index, ivar.values, lw=0.6, color="steelblue", label=r"$IV_t = VIX_t^2/12$")
ax.set_ylabel("$IV_t$ (monthly %²)")
ax.set_title("Step 1 — VIX-based Implied Variance (full history)")
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()

ax = axes[1]
ax.plot(ivar_vs.index,  ivar_vs.values,  lw=0.6, color="darkorange", label=r"$IV_t = VS_t^2/12$")
ax.plot(vix_ol.index, vix_ol.values, lw=0.6, color="steelblue", alpha=0.6, label=r"$IV_t = VIX_t^2/12$ (overlap)")
ax.set_ylabel("$IV_t$ (monthly %²)")
ax.set_title("Step 1 — VS-based Implied Variance vs VIX (overlap period)")
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()

plt.tight_layout()

---
## Step 2 — Physical Realized Variance Processing

Pull daily returns from `EquityFuture_historical.parquet`. Calculate the physical monthly realized variance using a rolling 22-day sum of squared daily returns:

| Component | Formula | Horizon |
|-----------|---------|----------|
| $RV_t^{(1)}$ | $(r_t \times 100)^2 \times 22$ | 1-day, monthly-scaled |
| $RV_t^{(5)}$ | $\frac{1}{5}\sum_{i=0}^{4}(r_{t-i} \times 100)^2 \times 22$ | 5-day rolling mean |
| $RV_t^{(22)}$ | $\sum_{i=0}^{21}(r_{t-i} \times 100)^2$ | 22-day rolling sum |

> **Data constraint:** Daily squared returns are ~252× noisier than 5-min intraday RV (~78 obs/day in the paper). This errors-in-variables effect inflates $\hat{\alpha}$ and attenuates lagged-RV coefficients.

In [ ]:
ret = load_sp500_returns()
rv  = compute_rv_components(ret)

print(f"SP500 returns: {len(ret):,} obs  "
      f"({ret.index.min().date()} – {ret.index.max().date()})")
print(rv.describe().round(3).to_string())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(ret.index, ret * 100, lw=0.4, color="steelblue", alpha=0.7)
axes[0].set_ylabel("Daily return (%)")
axes[0].set_title("SP500 E-mini Daily Returns")
axes[1].plot(rv.index, rv["RV22"], lw=0.7, color="darkorange",
             label=r"$RV_t^{(22)}$")
axes[1].plot(rv.index, rv["RV5"],  lw=0.5, color="green", alpha=0.7,
             label=r"$RV_t^{(5)}$")
axes[1].set_ylabel("RV (monthly %²)")
axes[1].set_title("Step 2 — Physical Realized Variance Components")
axes[1].legend()
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()

---
## Step 3 — Corsi HAR Panel Fitting (Bekaert & Hoerova, 2014)

Estimate an out-of-sample Corsi Heterogeneous Autoregressive (HAR) variance model, regressing the next-month realized variance against the recent 1-day, 5-day, and 22-day components:

$$CV_{t+1} = \underbrace{RV^{(22)}_{t+1}}_{\text{next-month realised var}} = c + \alpha \cdot IV_t + \beta^m \cdot RV_t^{(22)} + \beta^w \cdot RV_t^{(5)} + \beta^d \cdot RV_t^{(1)} + \varepsilon_t$$

Estimated via OLS with Newey-West HAC standard errors (44 lags). A 75% train / 25% test split establishes the **static baseline** against which the production loop is later compared.

In [ ]:
# ── Build VIX panel ────────────────────────────────────────────────────────────
panel = rv.join(ivar, how="inner").dropna()
panel["RV22_fwd"] = panel["RV22"].shift(-22)          # CV_{t+1}
panel["VIX2_lag"] = panel["IVar"].shift(1)             # IV_t  (har_model key)
panel["RV22_lag"] = panel["RV22"].shift(1)
panel["RV5_lag"]  = panel["RV5"].shift(1)
panel["RV1_lag"]  = panel["RV1"].shift(1)
panel = panel.dropna()

n = len(panel)
SPLIT_DATE = panel.index[int(0.75 * n)]

print(f"Panel: {n:,} obs  ({panel.index.min().date()} – {panel.index.max().date()})")
print(f"Train/test split (75%): {SPLIT_DATE.date()}  "
      f"(train = {int(0.75*n):,}, test = {n - int(0.75*n):,})")

In [ ]:
# ── HAR estimation ────────────────────────────────────────────────────────────
res_static = estimate_har(panel, "HAR-RV-VIX")
oos_static = out_of_sample_forecast(panel, SPLIT_DATE, "Static OOS")

print(f"In-Sample Adj-R²: {res_static['adj_r2']:.4f}   "
      f"IS RMSE: {res_static['rmse_is']:.3f}  "
      f"(paper: {PAPER_IS_RMSE}  ratio: {res_static['rmse_is']/PAPER_IS_RMSE:.1f}×)")
print()
print(f"{'Coefficient':<10} {'Ours':>10} {'NW-SE':>10} {'t-stat':>9} {'Paper':>10}")
print("-" * 55)
labels = {
    "const":    "c",
    "VIX2_lag": "α  (IV_t)",
    "RV22_lag": "β^m",
    "RV5_lag":  "β^w",
    "RV1_lag":  "β^d",
}
for v, lbl in labels.items():
    p  = res_static["params"][v]
    se = res_static["nw_se"][v]
    t  = res_static["t_stats"][v]
    pc = PAPER_COEFS.get(v, float("nan"))
    print(f"{lbl:<10} {p:>10.4f} {se:>10.4f} {t:>9.3f} {pc:>10.3f}")

---
## Step 4 — Variance Risk Premium Isolation

Compute the continuous daily premium vector via the difference between the risk-neutral expected variance (Step 1) and the physical conditional variance expectation (Step 3):

$$VP_t = IV_t - CV_t = \frac{VIX_t^2}{12} - \hat{RV}^{(22)}_{t+1}$$

- $VP_t > 0$ (~majority of days): investors pay a premium to offload variance risk
- $VP_t < 0$ (crisis peaks): realised variance spiked above the option-implied forecast

In [ ]:
panel["CV"] = res_static["fitted"]
panel["VP"] = panel["IVar"] - panel["CV"]

print(f"{'Stat':<10} {'VP':>10} {'CV':>10} {'IV_t':>10}")
print("-" * 45)
for stat in ["mean", "std", "min", "max"]:
    print(f"{stat.title():<10} "
          f"{getattr(panel['VP'],  stat)():>10.3f} "
          f"{getattr(panel['CV'],  stat)():>10.3f} "
          f"{getattr(panel['IVar'],stat)():>10.3f}")
print(f"{'% VP > 0':<10} {(panel['VP'] > 0).mean() * 100:>9.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.fill_between(panel.index, panel["VP"], 0,
                where=(panel["VP"] >= 0), color="steelblue", alpha=0.6,
                label=r"$VP_t > 0$")
ax.fill_between(panel.index, panel["VP"], 0,
                where=(panel["VP"] < 0), color="salmon", alpha=0.6,
                label=r"$VP_t < 0$")
ax.axhline(0, color="black", lw=0.7)
ax.set_ylabel(r"$VP_t = IV_t - CV_t$  (monthly %²)")
ax.set_title("Step 4 — Variance Risk Premium")
ax.legend()

ax = axes[1]
ax.plot(panel.index, panel["CV"],   color="darkorange", lw=0.8, label=r"$CV_t$  (HAR)")
ax.plot(panel.index, panel["IVar"], color="steelblue",  lw=0.6, alpha=0.5,
        label=r"$IV_t$  (VIX²/12)")
ax.set_ylabel("Variance (monthly %²)")
ax.set_title(r"$CV_t$ vs $IV_t$")
ax.legend()

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()

---
## Step 5 — End-of-Day Production Loop

Recalculate the rolling HAR components day by day and execute a **500-day rolling-window OLS** to dynamically update parameter weights and generate live $CV_t$ and $VP_t$ vectors:

1. At each day $t$: fit HAR on $[t-500,\; t-1]$ — parameters adapt to recent regime
2. Forecast $\hat{RV}^{(22)}_{t+1}$ with fresh coefficients → live $CV_t$
3. $VP_t = \text{IVVar}_t - CV_t$

Transitioning from retrospective static data to a latency-adjusted, rolling-window live feed prevents look-ahead bias and ensures the model organically **forgets stale structural regimes**.

> ⚠️ ~8,600 daily OLS fits — pre-computed results loaded here. Re-run via `experiment.py`.

In [ ]:
prod_full = pd.read_csv(OUTPUT / "production_loop_full.csv",
                         parse_dates=["date"]).set_index("date")
prod_oos  = prod_full[prod_full.index > SPLIT_DATE].copy()

print(f"Full rolling loop:  {len(prod_full):,} steps  "
      f"({prod_full.index.min().date()} – {prod_full.index.max().date()})")
print(f"OOS window:         {len(prod_oos):,} steps  "
      f"({prod_oos.index.min().date()} – {prod_oos.index.max().date()})")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(prod_oos.index, prod_oos["y_actual"], lw=0.6, color="steelblue",
        alpha=0.8, label=r"Actual $RV^{(22)}_{t+1}$")
ax.plot(prod_oos.index, prod_oos["y_hat"], lw=0.8, color="darkorange",
        label="500-day rolling HAR")
ax.set_ylabel("Monthly RV (%²)")
ax.set_title("Step 5 — Production Loop: Rolling HAR Forecast")
ax.legend()

ax = axes[1]
ax.plot(prod_oos.index, prod_oos["VP"], lw=0.7, color="steelblue",
        label=r"$VP_t$ (rolling)")
ax.plot(prod_oos.index, prod_oos["CV"], lw=0.7, color="darkorange",
        label=r"$CV_t$ (rolling)")
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("Variance (monthly %²)")
ax.set_title(r"Real-time $VP_t$ and $CV_t$ from Production Loop")
ax.legend()

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()

---
## Step 6 — Statistical Accuracy Evaluation (Bekaert & Hoerova, 2014)

Quantify the variance model's out-of-sample precision by computing RMSE, MAE, MAPE, and Mincer-Zarnowitz R². All metrics are evaluated on the same OOS window (post 75% split):

| Model | Estimation | Data |
|-------|-----------|------|
| **Static HAR** | One-time OLS on training set | VIX²/12, daily-sq RV |
| **Rolling HAR 500d** | Day-by-day refit on trailing 500 obs | VIX²/12, daily-sq RV |
| **Paper (B&H 2014)** | One-time OLS, 75% split | VIX²/12, 5-min RV |

The gap vs paper isolates the effect of the **daily-sq RV proxy** (~252× noisier than 5-min).  
The rolling vs static gap quantifies the benefit of **regime-adaptive re-estimation**.

In [ ]:
y_te_static  = oos_static["y_test"]
yhat_static  = oos_static["y_hat"]
common_idx   = y_te_static.index.intersection(prod_oos.index)

m_static  = compute_metrics(
    y_te_static.reindex(common_idx).values,
    yhat_static.reindex(common_idx).values,
    "Static HAR"
)
m_rolling = compute_metrics(
    prod_oos["y_actual"].reindex(common_idx).values,
    prod_oos["y_hat"].reindex(common_idx).values,
    "Rolling HAR 500d"
)

print(f"OOS window: {common_idx.min().date()} – {common_idx.max().date()}  "
      f"(n = {len(common_idx):,})")

cmp_df = pd.DataFrame([
    {"Model": "Static HAR",
     "MZ-R²": m_static["mz_r2"],  "RMSE": m_static["rmse"],
     "MAE":   m_static["mae"],    "MAPE": m_static["mape"]},
    {"Model": "Rolling HAR 500d",
     "MZ-R²": m_rolling["mz_r2"], "RMSE": m_rolling["rmse"],
     "MAE":   m_rolling["mae"],   "MAPE": m_rolling["mape"]},
    {"Model": "Paper (B&H 2014)",
     "MZ-R²": PAPER_OOS["mz_r2"], "RMSE": PAPER_OOS["rmse"],
     "MAE":   PAPER_OOS["mae"],   "MAPE": PAPER_OOS["mape"]},
]).set_index("Model")

display(cmp_df)

print(f"\nRolling vs Static   Δ MZ-R² = {m_rolling['mz_r2'] - m_static['mz_r2']:+.4f}  "
      f"Δ MAE = {m_rolling['mae'] - m_static['mae']:+.3f}")
print(f"Rolling vs Paper    RMSE ratio = {m_rolling['rmse'] / PAPER_OOS['rmse']:.2f}×  "
      f"MZ-R² gap = {m_rolling['mz_r2'] - PAPER_OOS['mz_r2']:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
models = ["Static\nHAR", "Rolling\nHAR 500d", "Paper\n(B&H)"]
colors = ["steelblue", "darkorange", "firebrick"]

for ax, metric, title, ylabel in zip(
    axes,
    ["mz_r2", "rmse", "mae"],
    ["OOS MZ-R²  (higher = better)",
     "OOS RMSE  (lower = better)",
     "OOS MAE  (lower = better)"],
    ["MZ-R²", "RMSE (%²)", "MAE (%²)"]
):
    vals = [m_static[metric], m_rolling[metric], PAPER_OOS[metric]]
    bars = ax.bar(models, vals, color=colors, alpha=0.8)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2,
                v + max(vals) * 0.01, f"{v:.3f}",
                ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

plt.suptitle("Step 6 — Statistical Accuracy Evaluation",
             fontsize=11, fontweight="bold")
plt.tight_layout()

---
## Summary

In [ ]:
# IS stats from the training set (same split used for OOS evaluation)
train_res = estimate_har(panel[panel.index <= SPLIT_DATE], "train IS")

summary_df = pd.DataFrame([
    {"Model":     "Static HAR",
     "IS Adj-R²": round(train_res["adj_r2"],   4),
     "IS RMSE":   round(train_res["rmse_is"],   3),
     "OOS MZ-R²": m_static["mz_r2"],
     "OOS RMSE":  m_static["rmse"],
     "OOS MAE":   m_static["mae"],
     "OOS MAPE":  m_static["mape"]},
    {"Model":     "Rolling HAR 500d",
     "IS Adj-R²": "—",   # no global IS fit; each day refits on trailing 500 days
     "IS RMSE":   "—",
     "OOS MZ-R²": m_rolling["mz_r2"],
     "OOS RMSE":  m_rolling["rmse"],
     "OOS MAE":   m_rolling["mae"],
     "OOS MAPE":  m_rolling["mape"]},
    {"Model":     "Paper (B&H 2014)",
     "IS Adj-R²": "—",
     "IS RMSE":   PAPER_IS_RMSE,
     "OOS MZ-R²": PAPER_OOS["mz_r2"],
     "OOS RMSE":  PAPER_OOS["rmse"],
     "OOS MAE":   PAPER_OOS["mae"],
     "OOS MAPE":  PAPER_OOS["mape"]},
]).set_index("Model")

display(summary_df)

### What explains the gap vs paper

The daily-sq RV proxy has ~252× higher estimation variance than 5-min intraday RV. This single data constraint explains:
- **$\hat{\alpha}$ inflated ~4–5×** vs paper's 0.108 — errors-in-variables: weight shifts onto the lower-noise forward-looking $IV_t$ signal
- **IS RMSE substantially higher** — noisy proxy inflates in-sample residuals

### OOS RMSE / MAE vs paper

Our OOS RMSE and MAE may appear lower than the paper's benchmarks (46.1 / 16.9). This is a **sample period effect, not a modelling improvement**:

- **Paper OOS window:** 2005–2010 — includes the 2008 GFC where $RV^{(22)}$ spiked to hundreds of %², making absolute errors very large
- **Our OOS window:** post-2021 — post-COVID normalisation and 2022 rate-hike vol, elevated but nowhere near GFC magnitude

RMSE and MAE are level-sensitive: a calmer OOS period mechanically produces lower absolute errors regardless of model quality. **MAPE** (which scales errors by actual RV) and **MZ-R²** (which measures forecast rank correlation) are the more comparable metrics across periods.

### Rolling vs Static

Δ MZ-R² and Δ MAE (Step 6) quantify the regime-adaptation benefit of 500-day rolling re-estimation. On this panel, the static model wins because estimation precision loss from the short rolling window outweighs the regime-forgetting benefit — see the rolling vs static discussion in the notebook.

---
## VIX-based vs VS-based VRP Comparison

Run a parallel production loop using pure VS²/12 implied variance (2008 onwards) and compare the resulting VRP series and HAR forecast quality against the VIX-based loop.

In [ ]:
# Load pre-computed production loop results from experiment.py
prod_vix = pd.read_csv(OUTPUT / "production_loop_full.csv",
                       parse_dates=["date"]).set_index("date")
prod_vs  = pd.read_csv(OUTPUT / "production_loop_vs.csv",
                       parse_dates=["date"]).set_index("date")

overlap_start = prod_vs.index.min()
overlap_end   = min(prod_vix.index.max(), prod_vs.index.max())
vix_ol = prod_vix.loc[overlap_start:overlap_end]
vs_ol  = prod_vs.loc[overlap_start:overlap_end]
corr   = vix_ol["VP"].corr(vs_ol["VP"].reindex(vix_ol.index))

print(f"Overlap: {overlap_start.date()} - {overlap_end.date()}  ({len(vs_ol):,} obs)")
print()
print(f"{'Statistic':<25} {'VIX-VRP':>12} {'VS-VRP':>12}")
print("-" * 52)
for stat, fn in [("Mean VP", "mean"), ("Median VP", "median"), ("Std VP", "std")]:
    v = getattr(vix_ol["VP"], fn)()
    s = getattr(vs_ol["VP"],  fn)()
    print(f"{stat:<25} {v:>12.3f} {s:>12.3f}")
print(f"{'% VP > 0':<25} {(vix_ol['VP']>0).mean()*100:>11.1f}% {(vs_ol['VP']>0).mean()*100:>11.1f}%")
print(f"{'Corr(VIX-VRP, VS-VRP)':<25} {corr:>12.4f}")
print()
print(f"{'IVar mean (VIX)':<25} {vix_ol['IVar'].mean():>12.3f}")
print(f"{'IVar mean (VS)':<25} {vs_ol['IVar'].mean():>12.3f}")
print(f"{'IVar ratio (VIX/VS)':<25} {vix_ol['IVar'].mean()/vs_ol['IVar'].mean():>12.4f}")

In [ ]:
# Load monthly MZ-R² diagnostics for both loops
diag_vix = pd.read_csv(OUTPUT / "dm_diagnostics_full.csv",
                       parse_dates=["date"]).set_index("date")
diag_vs  = pd.read_csv(OUTPUT / "dm_diagnostics_vs.csv",
                       parse_dates=["date"]).set_index("date")
diag_vix_ol = diag_vix[diag_vix.index >= overlap_start]
diag_vs_ol  = diag_vs[diag_vs.index  >= overlap_start]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

# ── Panel 1: VRP time series ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(vix_ol.index, vix_ol["VP"], color="steelblue",  lw=0.7, alpha=0.9,
        label=f"VRP (VIX²/12)  mean={vix_ol['VP'].mean():.2f}")
ax.plot(vs_ol.index,  vs_ol["VP"],  color="darkorange", lw=0.7, alpha=0.9,
        label=f"VRP (VS²/12)   mean={vs_ol['VP'].mean():.2f}")
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("VP = IVar - CV  (%² monthly)")
ax.set_title(f"VRP: VIX²/12 vs VS²/12  |  Overlap {overlap_start.date()} - {overlap_end.date()}"
             f"  |  Corr={corr:.3f}")
ax.legend(fontsize=9)
ax.set_ylim(-250, 400)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# ── Panel 2: IVar time series ─────────────────────────────────────────────────
ax = axes[1]
ax.plot(vix_ol.index, vix_ol["IVar"], color="steelblue",  lw=0.6, alpha=0.85,
        label=f"IVar (VIX²/12)  mean={vix_ol['IVar'].mean():.2f}")
ax.plot(vs_ol.index,  vs_ol["IVar"],  color="darkorange", lw=0.6, alpha=0.85,
        label=f"IVar (VS²/12)   mean={vs_ol['IVar'].mean():.2f}")
ax.set_ylabel("Implied Variance (%² monthly)")
ax.set_title(f"Implied Variance  |  VIX/VS ratio = {vix_ol['IVar'].mean()/vs_ol['IVar'].mean():.3f}")
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# ── Panel 3: Monthly MZ-R² ────────────────────────────────────────────────────
ax = axes[2]
ax.plot(diag_vix_ol.index, diag_vix_ol["mz_r2"], color="steelblue",  lw=1.2,
        marker="o", markersize=2, label=f"MZ-R² (VIX-HAR)  mean={diag_vix_ol['mz_r2'].mean():.3f}")
ax.plot(diag_vs_ol.index,  diag_vs_ol["mz_r2"],  color="darkorange", lw=1.2,
        marker="o", markersize=2, label=f"MZ-R² (VS-HAR)   mean={diag_vs_ol['mz_r2'].mean():.3f}")
ax.axhline(0, color="black", lw=0.4)
ax.set_ylabel("Monthly MZ-R²  (12-month trailing)")
ax.set_title("HAR Forecast Quality: VIX-based vs VS-based")
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()